# TACO-X Inference Demo

Interactive benchmark notebook for TACO-X (and vLLM). Run each cell to execute benchmark steps one at a time, inspect results, and view server logs.

**Prerequisites**: Server running at the configured `BASE_URL` (TACO-X on port 18080 or vLLM on port 8000).

## 1. Setup & Connection

In [18]:
# Install openai if missing (run this cell once, then run the cell below)
%pip install -q openai

Note: you may need to restart the kernel to use updated packages.


In [19]:
import asyncio
import json
import time
import subprocess
from IPython.display import display, HTML, Markdown, clear_output
from openai import OpenAI, AsyncOpenAI

# ── Configuration ──
BASE_URL = "http://124.156.210.57:18080/v1"  # TACO-X
# BASE_URL = "http://124.156.210.57:8000/v1"  # vLLM
MODEL = "/data2/models/Qwen3-32B"
MAX_TOKENS = 256

client = OpenAI(base_url=BASE_URL, api_key="not-needed")
aclient = AsyncOpenAI(base_url=BASE_URL, api_key="not-needed")

print(f"Server:     {BASE_URL}")
print(f"Model:      {MODEL}")
print(f"Max tokens: {MAX_TOKENS}")

Server:     http://124.156.210.57:18080/v1
Model:      /data2/models/Qwen3-32B
Max tokens: 256


## 2. Health Check

In [20]:
# Health check — try /v1/models, fall back gracefully for TACO-X
try:
    models = client.models.list()
    print("Available models:")
    for m in models.data:
        print(f"  - {m.id}")
        MODEL = m.id
except Exception:
    print(f"Note: /v1/models not supported (TACO-X). Using MODEL={MODEL}")

# Quick inference check
try:
    r = client.chat.completions.create(model=MODEL, messages=[{'role':'user','content':'hi'}], max_tokens=5)
    print(f"Server OK — got: {r.choices[0].message.content[:50]}")
except Exception as e:
    print(f"Server error: {e}")

Note: /v1/models not supported (TACO-X). Using MODEL=/data2/models/Qwen3-32B
Server OK — got: <think>
Okay, the


## 3. Prompts

In [21]:
SYSTEM_PROMPT = "You are a helpful assistant."

PROMPTS = {
    "short": "Explain cloud computing in simple terms. Cover what it is, why businesses use it, and give two real-world examples.",
    "medium": (
        "Analyze the following product description and provide a competitive assessment.\n\n"
        "Product: CloudSync Pro - Enterprise File Synchronization Platform\n\n"
        "CloudSync Pro is an enterprise-grade file synchronization and sharing platform "
        "designed for organizations with 500+ employees. The platform offers real-time "
        "file synchronization across Windows, macOS, Linux, iOS, and Android devices. "
        "Key features include end-to-end AES-256 encryption, granular access controls "
        "with role-based permissions, and comprehensive audit logging for compliance.\n\n"
        "Questions: 1) What are the three strongest competitive advantages? "
        "2) What gaps might enterprise buyers identify? "
        "3) How does the pricing compare to Dropbox Business and Box Enterprise?"
    ),
    "long": (
        "Summarize the following article and extract the five most important takeaways.\n\n"
        "Article: The Evolution of Distributed Systems Architecture\n\n"
        "The history of distributed systems architecture spans over five decades, from "
        "the earliest time-sharing systems of the 1960s to today's globally distributed "
        "cloud-native applications. In the 1970s, the emergence of ARPANET introduced "
        "networked computing. The 1980s and 1990s saw client-server architecture dominate. "
        "The early 2000s brought SOA. REST APIs gained adoption in the mid-2000s. "
        "Microservices emerged around 2011-2014, popularized by Netflix and Amazon. "
        "Docker (2013) and Kubernetes (2014) provided the infrastructure foundation. "
        "Serverless computing pushed abstraction further with AWS Lambda (2014). "
        "Event-driven architecture with Apache Kafka became the backbone of many systems. "
        "Service meshes like Istio address cross-cutting concerns. Looking forward, "
        "edge computing, WebAssembly, and AIOps are shaping the next generation. "
        "The fundamental challenges of distributed systems remain as relevant as ever."
    ),
}

for name, prompt in PROMPTS.items():
    print(f"{name:>8}: {len(prompt)} chars, ~{len(prompt)//4} tokens (est)")

   short: 115 chars, ~28 tokens (est)
  medium: 722 chars, ~180 tokens (est)
    long: 1057 chars, ~264 tokens (est)


## 4. Helper Functions

In [22]:
def send_streaming(prompt, max_tokens=MAX_TOKENS, temperature=0, show_output=True):
    """Send a single streaming request. Returns metrics dict."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    t_start = time.perf_counter()
    ttft_ms = None
    prompt_tokens = 0
    completion_tokens = 0
    text = ""

    stream = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=temperature,
        max_tokens=max_tokens, stream=True,
        stream_options={"include_usage": True},
    )

    for chunk in stream:
        if hasattr(chunk, "usage") and chunk.usage is not None:
            prompt_tokens = chunk.usage.prompt_tokens
            completion_tokens = chunk.usage.completion_tokens
            continue
        if not chunk.choices:
            continue
        if ttft_ms is None:
            ttft_ms = (time.perf_counter() - t_start) * 1000
        delta = chunk.choices[0].delta.content
        if delta:
            text += delta
            if show_output:
                print(delta, end="", flush=True)

    e2e_s = time.perf_counter() - t_start
    tpot_ms = ((e2e_s * 1000) - ttft_ms) / (completion_tokens - 1) if completion_tokens > 1 and ttft_ms else None
    tok_s = completion_tokens / e2e_s if e2e_s > 0 else 0

    if show_output:
        print()

    metrics = {
        "ttft_ms": round(ttft_ms, 1) if ttft_ms else None,
        "tpot_ms": round(tpot_ms, 1) if tpot_ms else None,
        "e2e_s": round(e2e_s, 3),
        "tok_s": round(tok_s, 1),
        "prompt_tokens": prompt_tokens,
        "completion_tokens": completion_tokens,
    }
    return metrics, text


def metrics_table(results):
    """Display a list of metrics dicts as an HTML table."""
    rows = ""
    for r in results:
        rows += f"<tr><td>{r.get('label','')}</td><td>{r['ttft_ms']}</td><td>{r['tpot_ms']}</td><td>{r['e2e_s']}</td><td><b>{r['tok_s']}</b></td><td>{r['prompt_tokens']}</td><td>{r['completion_tokens']}</td></tr>"
    html = f"""
    <table style="border-collapse:collapse; font-family:monospace; font-size:13px;">
    <tr style="background:#f0f0f0;"><th style="padding:6px 12px;">Test</th><th>TTFT (ms)</th><th>TPOT (ms)</th><th>E2E (s)</th><th>Tok/s</th><th>Prompt</th><th>Output</th></tr>
    {rows}
    </table>"""
    display(HTML(html.replace("<td>", '<td style="padding:4px 12px; border-bottom:1px solid #ddd;">')))


def get_docker_logs(container="taco_x", lines=50):
    """Get recent docker logs from the TACO-X container."""
    try:
        out = subprocess.check_output(
            ["sudo", "docker", "logs", container, "--tail", str(lines)],
            text=True, stderr=subprocess.STDOUT, timeout=5
        )
        return out
    except Exception as e:
        return f"Could not get logs: {e}"


print("Helper functions loaded.")

Helper functions loaded.


## 5. Warmup — Populate Lookahead Cache

Send warmup prompts to pre-populate the lookahead cache before benchmarking.

In [23]:
warmup_prompts = [
    "Explain cloud computing in simple terms.",
    "What are the main differences between TCP and UDP?",
    "Describe the key principles of microservices architecture.",
    "What is containerization? How do Docker and Kubernetes work together?",
    "Explain the CAP theorem in distributed systems.",
]

print(f"Sending {len(warmup_prompts)} warmup prompts...")
for i, wp in enumerate(warmup_prompts, 1):
    m, _ = send_streaming(wp, show_output=False)
    print(f"  [{i}/{len(warmup_prompts)}] {m['completion_tokens']} tok, {m['e2e_s']}s, {m['tok_s']} tok/s")
print("Warmup complete.")

Sending 5 warmup prompts...
  [1/5] 256 tok, 7.106s, 36.0 tok/s
  [2/5] 256 tok, 4.503s, 56.8 tok/s
  [3/5] 256 tok, 4.248s, 60.3 tok/s
  [4/5] 256 tok, 3.123s, 82.0 tok/s
  [5/5] 256 tok, 6.55s, 39.1 tok/s
Warmup complete.


## 6. Benchmark — Single Request Sweep

Run each prompt size at concurrency 1, collecting metrics.

In [ ]:
NUM_REQUESTS = 5  # requests per config (keep low for demo)

results = []
for size in ["short", "medium", "long"]:
    ttfts, tpots, e2es, tok_ss = [], [], [], []
    print(f"Running {size}...", end=" ", flush=True)
    for i in range(NUM_REQUESTS):
        m, _ = send_streaming(PROMPTS[size], show_output=False)
        if m["ttft_ms"]: ttfts.append(m["ttft_ms"])
        if m["tpot_ms"]: tpots.append(m["tpot_ms"])
        e2es.append(m["e2e_s"])
        tok_ss.append(m["tok_s"])
    
    import statistics
    result = {
        "label": f"{size} (c=1)",
        "ttft_ms": round(statistics.median(ttfts), 1) if ttfts else None,
        "tpot_ms": round(statistics.median(tpots), 1) if tpots else None,
        "e2e_s": round(statistics.median(e2es), 2),
        "tok_s": round(statistics.median(tok_ss), 1),
        "prompt_tokens": m["prompt_tokens"],
        "completion_tokens": m["completion_tokens"],
    }
    results.append(result)
    print(f"{result['tok_s']} tok/s, TTFT {result['ttft_ms']}ms, TPOT {result['tpot_ms']}ms")

print()
metrics_table(results)

## 7. Benchmark — Concurrency Test

Run multiple requests in parallel to test throughput under load.

In [ ]:
import asyncio

async def send_async(prompt, max_tokens=MAX_TOKENS):
    """Send a single async streaming request."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    t_start = time.perf_counter()
    ttft_ms = None
    prompt_tokens = completion_tokens = 0

    stream = await aclient.chat.completions.create(
        model=MODEL, messages=messages, temperature=0,
        max_tokens=max_tokens, stream=True,
        stream_options={"include_usage": True},
    )
    async for chunk in stream:
        if hasattr(chunk, "usage") and chunk.usage is not None:
            prompt_tokens = chunk.usage.prompt_tokens
            completion_tokens = chunk.usage.completion_tokens
            continue
        if not chunk.choices:
            continue
        if ttft_ms is None:
            ttft_ms = (time.perf_counter() - t_start) * 1000

    e2e_s = time.perf_counter() - t_start
    tpot_ms = ((e2e_s * 1000) - ttft_ms) / (completion_tokens - 1) if completion_tokens > 1 and ttft_ms else None
    tok_s = completion_tokens / e2e_s if e2e_s > 0 else 0
    return {"ttft_ms": round(ttft_ms, 1) if ttft_ms else None, "tpot_ms": round(tpot_ms, 1) if tpot_ms else None, "e2e_s": round(e2e_s, 3), "tok_s": round(tok_s, 1), "prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens}


async def run_concurrent(prompt, concurrency, num_requests=5):
    """Run num_requests at given concurrency."""
    sem = asyncio.Semaphore(concurrency)
    async def limited():
        async with sem:
            return await send_async(prompt)
    tasks = [limited() for _ in range(num_requests)]
    return await asyncio.gather(*tasks)


conc_results = []
for conc in [1, 5, 10]:
    print(f"Running short prompt at concurrency {conc}...", end=" ", flush=True)
    batch = await run_concurrent(PROMPTS["short"], conc, num_requests=5)
    ttfts = [r["ttft_ms"] for r in batch if r["ttft_ms"]]
    tpots = [r["tpot_ms"] for r in batch if r["tpot_ms"]]
    tok_ss = [r["tok_s"] for r in batch]
    result = {
        "label": f"short (c={conc})",
        "ttft_ms": round(statistics.median(ttfts), 1) if ttfts else None,
        "tpot_ms": round(statistics.median(tpots), 1) if tpots else None,
        "e2e_s": round(statistics.median([r["e2e_s"] for r in batch]), 2),
        "tok_s": round(statistics.median(tok_ss), 1),
        "prompt_tokens": batch[0]["prompt_tokens"],
        "completion_tokens": batch[0]["completion_tokens"],
    }
    conc_results.append(result)
    print(f"{result['tok_s']} tok/s, TPOT {result['tpot_ms']}ms")

print()
metrics_table(conc_results)

## 8. Multi-Turn Conversation

Test how the engine handles growing context (prefix caching).

In [ ]:
conversation = [
    "What are the main differences between TCP and UDP?",
    "Can you give me a real-world example where UDP is preferred over TCP?",
    "How does QUIC improve on both TCP and UDP?",
    "Summarize the trade-offs between TCP, UDP, and QUIC in a table format.",
    "Based on everything we discussed, what would you recommend for a video streaming application and why?",
]

messages = [{"role": "system", "content": SYSTEM_PROMPT}]
turn_results = []

for turn, user_msg in enumerate(conversation, 1):
    messages.append({"role": "user", "content": user_msg})
    print(f"--- Turn {turn}: {user_msg[:60]}... ---")
    
    t_start = time.perf_counter()
    ttft_ms = None
    text = ""
    prompt_tokens = completion_tokens = 0
    
    stream = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=0,
        max_tokens=MAX_TOKENS, stream=True,
        stream_options={"include_usage": True},
    )
    for chunk in stream:
        if hasattr(chunk, "usage") and chunk.usage is not None:
            prompt_tokens = chunk.usage.prompt_tokens
            completion_tokens = chunk.usage.completion_tokens
            continue
        if not chunk.choices:
            continue
        if ttft_ms is None:
            ttft_ms = (time.perf_counter() - t_start) * 1000
        delta = chunk.choices[0].delta.content
        if delta:
            text += delta
    
    e2e_s = time.perf_counter() - t_start
    tpot_ms = ((e2e_s * 1000) - ttft_ms) / (completion_tokens - 1) if completion_tokens > 1 and ttft_ms else None
    tok_s = completion_tokens / e2e_s if e2e_s > 0 else 0
    
    messages.append({"role": "assistant", "content": text})
    
    r = {"label": f"Turn {turn} ({prompt_tokens} tok in)", "ttft_ms": round(ttft_ms, 1) if ttft_ms else None, "tpot_ms": round(tpot_ms, 1) if tpot_ms else None, "e2e_s": round(e2e_s, 3), "tok_s": round(tok_s, 1), "prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens}
    turn_results.append(r)
    print(f"  TTFT: {r['ttft_ms']}ms | TPOT: {r['tpot_ms']}ms | {r['tok_s']} tok/s | {prompt_tokens} prompt tok")

print()
metrics_table(turn_results)

## 9. Custom Prompt

Type your own prompt and see streaming output with metrics.

In [ ]:
# Edit this prompt:
my_prompt = "What are the top 3 benefits of using GPU inference acceleration for large language models?"

print(f"Prompt: {my_prompt}\n")
print("=" * 60)
metrics, text = send_streaming(my_prompt, max_tokens=512)
print("=" * 60)
print()
for k, v in metrics.items():
    print(f"  {k:>20}: {v}")